In [2]:
# El Caso del Historial Perdido
# Wilcoxon pareado exacto de E1 a E4 contra el baseline E0.
# Cada fila es una semilla. Cada valor es el mejor de sus 7 cortes.

maximos <- data.frame(
  semilla = c(915851, 884951, 687121, 664211, 616111,
              999853, 999863, 999883, 999907, 999917),

  E0 = c(83.655, 90.301, 89.138, 86.729, 86.147,
         80.415, 89.968, 84.319, 84.070, 90.633),

  E1 = c(81.578, 83.406, 80.747, 83.323, 88.722,
         81.993, 80.996, 87.809, 82.243, 84.403),

  E2 = c(88.889, 79.667, 86.562, 80.913, 84.652,
         79.086, 80.996, 82.575, 76.926, 87.393),

  E3 = c(79.335, 81.495, 81.661, 81.993, 87.310,
         84.319, 87.892, 80.664, 83.821, 87.144),

  E4 = c(84.236, 80.249, 83.572, 81.080, 81.412,
         76.178, 75.347, 83.738, 78.255, 78.255)
)

experimentos <- paste0("E", 0:4)

stopifnot(
  nrow(maximos) == 10L,
  anyDuplicated(maximos$semilla) == 0L,
  all(is.finite(as.matrix(maximos[experimentos])))
)

# Prueba exacta de rangos con signo.
# Los empates en magnitud reciben el rango promedio.
wilcoxon_exacto <- function(diferencias) {
  diferencias <- diferencias[diferencias != 0]

  if (length(diferencias) == 0L) {
    return(c(p_bilateral = 1, p_mejora = 1, p_caida = 1))
  }

  rangos <- rank(abs(diferencias), ties.method = "average")
  observado <- sum(rangos[diferencias > 0])

  signos <- expand.grid(
    rep(list(c(0L, 1L)), length(rangos)),
    KEEP.OUT.ATTRS = FALSE
  )

  distribucion_nula <- as.vector(as.matrix(signos) %*% rangos)
  p_mejora <- mean(distribucion_nula >= observado)
  p_caida <- mean(distribucion_nula <= observado)

  c(
    p_bilateral = min(1, 2 * min(p_mejora, p_caida)),
    p_mejora = p_mejora,
    p_caida = p_caida
  )
}

resumen <- data.frame(
  experimento = experimentos,
  variante = c("Baseline (9100)", "Serie 9200", "Tendencias (9300)",
               "Medias móviles", "Todo junto (9400)"),
  n_semillas = nrow(maximos),
  media_maximos = unname(colMeans(maximos[experimentos])),
  diferencia_mediana_vs_E0 = NA_real_,
  semillas_que_mejoran = NA_integer_,
  p_bilateral = NA_real_,
  p_mejora = NA_real_,
  p_caida = NA_real_
)

for (i in 2:nrow(resumen)) {
  variante <- maximos[[resumen$experimento[i]]]

  # Comparar variante − E0 para la misma semilla.
  # Usar tres decimales para reconocer los empates a la precisión original.
  diferencias <- round(variante - maximos$E0, digits = 3)
  prueba <- wilcoxon_exacto(diferencias)

  resumen$diferencia_mediana_vs_E0[i] <- median(diferencias)
  resumen$semillas_que_mejoran[i] <- sum(diferencias > 0)
  resumen$p_bilateral[i] <- prueba[["p_bilateral"]]
  resumen$p_mejora[i] <- prueba[["p_mejora"]]
  resumen$p_caida[i] <- prueba[["p_caida"]]
}

# Redondear solo la tabla que se muestra y se exporta.
tabla <- resumen
tabla$media_maximos <- round(tabla$media_maximos, 2)
tabla$diferencia_mediana_vs_E0 <- round(tabla$diferencia_mediana_vs_E0, 2)

columnas_p <- c("p_bilateral", "p_mejora", "p_caida")
tabla[columnas_p] <- lapply(tabla[columnas_p], round, digits = 4)

dir.create("resultados", showWarnings = FALSE)
archivo <- file.path("resultados", "resumen_wilcoxon.csv")

write.csv(
  tabla,
  file = archivo,
  row.names = FALSE,
  na = "",
  fileEncoding = "UTF-8"
)

print(tabla, row.names = FALSE, na.print = "—")
message("Tabla guardada en ", normalizePath(archivo))

 experimento          variante n_semillas media_maximos
          E0   Baseline (9100)         10         86.54
          E1        Serie 9200         10         83.52
          E2 Tendencias (9300)         10         82.77
          E3    Medias móviles         10         83.56
          E4 Todo junto (9400)         10         80.23
 diferencia_mediana_vs_E0 semillas_que_mejoran p_bilateral p_mejora p_caida
                       NA                   NA          NA       NA      NA
                    -2.74                    3      0.1055   0.9580  0.0527
                    -2.91                    1      0.0273   0.9902  0.0137
                    -3.57                    2      0.0488   0.9814  0.0244
                    -5.61                    1      0.0059   0.9990  0.0029


Tabla guardada en /home/ds/buckets/b1/labo2026v/ExpColaborativos/resultados/resumen_wilcoxon.csv

